# BTP step S1b on Kaggle: TL-HF equivalence checks (plan Table 5)

Runs `scripts/s1_equivalence.sh gemma-2-2b-it cuda 2` on Kaggle's **GPU T4 x2**: the HF copy runs on `cuda:0` first, then moves to CPU, and the TransformerLens copy is split over both GPUs.

**Before running**
1. Settings: Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified account).
2. Add the private dataset `btp-code` (holding `btp_code.zip`) as input.
3. Gemma is gated: accept the licence at huggingface.co/google/gemma-2-2b-it, then add a Kaggle secret (Add-ons > Secrets) named `HF_TOKEN` and attach it to this notebook.

S1c (`s1_sae.sh`) is deliberately not here: `prereg/E0.md` must be reviewed and committed first.

In [ ]:
!nvidia-smi
!python --version
import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Need 2 GPUs: set Accelerator to GPU T4 x2'

In [ ]:
# Find the code under /kaggle/input: Kaggle may keep btp_code.zip or auto-extract it into a folder.
import glob, os, shutil, zipfile
DST = '/kaggle/working/Reasoning-Failure-Diagnosis'
shutil.rmtree(DST, ignore_errors=True)
zips = glob.glob('/kaggle/input/**/btp_code.zip', recursive=True)
reqs = [p for p in glob.glob('/kaggle/input/**/requirements.txt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
if zips:
    print('zip:', zips[0])
    tmp = '/kaggle/working/_unzip'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zips[0]).extractall(tmp)
    inner = [os.path.dirname(p) for p in glob.glob(tmp + '/**/requirements.txt', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
    assert inner, 'no project root inside the zip'
    shutil.copytree(min(inner, key=len), DST)
    shutil.rmtree(tmp)
elif reqs:
    src = os.path.dirname(min(reqs, key=len))
    print('folder:', src)
    shutil.copytree(src, DST)
else:
    raise FileNotFoundError('btp_code not found under /kaggle/input; add the btp-code dataset as input')
assert 'numpy>=2.0' in open(f'{DST}/requirements.txt').read(), 'stale requirements.txt'
print(sorted(os.listdir(DST)))
assert 'hf_reference' in open(f'{DST}/harness/equivalence.py').read(), 'old btp-code version: update the dataset input'

In [ ]:
# requirements.txt needs Python 3.13 (TransformerLens 2.x caps numpy<2 on <=3.12), so build a 3.13 venv with uv.
!pip -q install uv
!uv venv --seed --python 3.13 /kaggle/working/venv
!uv pip install --python /kaggle/working/venv/bin/python -r /kaggle/working/Reasoning-Failure-Diagnosis/requirements.txt
os.environ['PATH'] = '/kaggle/working/venv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/kaggle/working/venv'
!which python && python -c "import torch, transformer_lens, numpy; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), numpy.__version__)"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')   # never printed
print('HF_TOKEN set:', bool(os.environ['HF_TOKEN']))

In [ ]:
!cd /kaggle/working/Reasoning-Failure-Diagnosis && python -m pytest -q

In [ ]:
!cd /kaggle/working/Reasoning-Failure-Diagnosis && bash scripts/s1_equivalence.sh gemma-2-2b-it cuda 2

In [ ]:
# Copy results to /kaggle/working/reports_m0 (downloadable from the Output tab) and summarise.
import json
OUT = '/kaggle/working/reports_m0'
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree('/kaggle/working/Reasoning-Failure-Diagnosis/reports/m0', OUT)
print(sorted(os.listdir(OUT)))
for p in sorted(glob.glob(OUT + '/equivalence_*.json')):
    r = json.load(open(p))
    print(r['model'], r['device'], 'tl_n_devices', r.get('tl_n_devices'), r['dtype'], 'torch', r['torch'], r.get('tl_soft_caps'))
    print('logits_rel_err', r['logits_rel_err'])
    for k, v in r['per_layer_rel_err'].items():
        print(f"  {k}: {v:.3e}{'  <-- above 1e-4' if v > 1e-4 else ''}")
    for name, c in r['checks'].items():
        print(f"{'PASS' if c['pass'] else 'FAIL'}  {name}: {c['value']}  ({c.get('threshold', '')})")
    print('ALL PASS' if r['all_pass'] else 'SOME CHECKS FAILED')